# Synthetic Tabular Data Generation - Streamlined Driver

**Streamlined notebook for synthetic tabular data generation with batch processing.**

This notebook provides a config-driven workflow for:
- Section 1: Setup and imports
- Section 2: Data loading, preprocessing, and EDA
- Section 3: Demo models with default parameters
- Section 4: Hyperparameter optimization
- Section 5: Final models with best parameters

## 1 Setup and Configuration

In [2]:
# Code Chunk ID: CHUNK_1_0_0_A - Import Setup Module
# Import all functionality from setup.py
import sys
sys.path.insert(0, "/home/ec2-user/SageMaker/tableGenCompare")

from setup import *

# Refresh session timestamp to current date
refresh_session_timestamp()

print("SETUP MODULE IMPORTED SUCCESSFULLY!")
print("="*60)
import sys                                                                                                                                                                                                     
!{sys.executable} -m pip install ctgan sdv ganeraid 

[OK] Essential data science libraries imported successfully!
[OK] Model registry loaded from src/models/registry.py
[OK] Batch training module loaded from src/models/batch_training.py
[OK] Search spaces module loaded from src/models/search_spaces.py
[OK] Batch optimization module loaded from src/models/batch_optimization.py
[CONFIG] Session timestamp: 2026-01-22
[OK] Parameter management functions loaded from src/utils/parameters.py
[OK] Hyperparameter optimization evaluation functions loaded from src/evaluation/hyperparameters.py
[OK] Optuna objective functions for PATE-GAN and MEDGAN loaded (Phase 5)
Detected sklearn 1.7.2 - applying compatibility patch...
INFO: Applying sklearn compatibility patches for version 1.7.2
Global sklearn compatibility patch applied successfully
CTAB-GAN imported successfully
[OK] CTAB-GAN+ detected and available
[OK] GANerAidModel imported successfully from src.models.implementations
[OK] Config-driven preprocessing functions loaded (Phase 3)
[OK] EDA mod

## 2 Data Loading and Pre-processing

### 2.1 Configuration

**USER ATTENTION NEEDED**: Edit the `NOTEBOOK_CONFIG` dictionary below to match your dataset.

In [9]:
# Code Chunk ID: CHUNK_2_1_1_A - NOTEBOOK CONFIGURATION
# ============================================================================
# USER CONFIGURATION - Edit ONLY this block for your dataset
# ============================================================================

NOTEBOOK_CONFIG = {
    # ========== REQUIRED: Dataset Settings ==========
    "data_file": "data/Pakistani_Diabetes_Dataset.csv",  # Path to your CSV file
    "target_column": "Outcome",                 # Target/outcome column name

    # ========== OPTIONAL: Dataset Metadata ==========
    "dataset_name": "Pakistani Diabetes Dataset",      # Display name
    "dataset_identifier_override": None,          # Override auto-detected ID (or None)

    # ========== OPTIONAL: Column Configuration ==========
    "categorical_columns": ["gender", "Rgn"],                    # List categorical cols, or [] for auto-detect
    "task_type": "auto",                          # "auto" | "classification" | "regression"

    # ========== OPTIONAL: Data Subsetting ==========
    "use_row_subset": False,                      # True to sample rows for faster testing
    "sample_n": 1500,                              # Number of rows to sample
    "sample_random_state": 42,                    # Random seed for reproducibility

    # ========== OPTIONAL: Missing Data Handling ==========
    "missing_strategy": "mice",                   # "none" | "drop" | "median" | "mode" | "mice"
    "mice_max_iter": 10,                          # Max iterations for MICE imputation

    # ========== OPTIONAL: Model Selection ==========
    "models_to_run": "all",                       # "all" or list like ["ctgan", "tvae"]

    # ========== OPTIONAL: Tuning Configuration ==========
    "tuning_mode": "smoke",                        # "smoke" (quick) | "full" (thorough)
    "n_trials_smoke": 5,                          # Trials for smoke testing
    "n_trials_full": 30,                          # Trials for full optimization
}

print("NOTEBOOK_CONFIG set successfully!")
print(f"  Data file: {NOTEBOOK_CONFIG['data_file']}")
print(f"  Target column: {NOTEBOOK_CONFIG['target_column']}")
print(f"  Models to run: {NOTEBOOK_CONFIG['models_to_run']}")
print(f"  Tuning mode: {NOTEBOOK_CONFIG['tuning_mode']}")

NOTEBOOK_CONFIG set successfully!
  Data file: data/Pakistani_Diabetes_Dataset.csv
  Target column: Outcome
  Models to run: all
  Tuning mode: smoke


### 2.2 Load and Preprocess Data

In [10]:
# Code Chunk ID: CHUNK_2_1_2_A - LOAD AND PREPROCESS DATA
# ============================================================================
# This uses the config-driven preprocessing pipeline
# ============================================================================

# Load and preprocess data using the config
(
    data,                  # Processed DataFrame
    original_data,         # Copy for reference
    target_column,         # Target column name (cleaned)
    DATASET_IDENTIFIER,    # Dataset identifier for results paths
    categorical_columns,   # List of categorical columns
    metadata               # Full preprocessing metadata
) = load_and_preprocess_from_config(NOTEBOOK_CONFIG)

# Set aliases for backward compatibility with existing notebook cells
TARGET_COLUMN = target_column

# Get models to run based on config
models_to_run = get_models_to_run(NOTEBOOK_CONFIG)
tuning_config = get_tuning_config(NOTEBOOK_CONFIG)
N_TRIALS = get_n_trials(NOTEBOOK_CONFIG)

# Set RUN_MODE for backward compatibility with Section 4 tuning cells
RUN_MODE = "debug" if NOTEBOOK_CONFIG['tuning_mode'] == "smoke" else "full"

print()
print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"  Dataset identifier: {DATASET_IDENTIFIER}")
print(f"  Processed shape: {data.shape}")
print(f"  Target column: {target_column}")
print(f"  Task type: {metadata['task_type']}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Models to run: {models_to_run}")
print(f"  RUN_MODE: {RUN_MODE}")
print(f"  N_TRIALS: {N_TRIALS}")
print(f"  Session: {SESSION_TIMESTAMP}")
print(f"  Results path: {get_results_path(DATASET_IDENTIFIER, 2)}")
print("=" * 60)

[LOAD] Loading data from: data/Pakistani_Diabetes_Dataset.csv
[LOAD] Loaded 912 rows, 19 columns
[PREPROCESS] Starting preprocessing pipeline
[PREPROCESS] Input shape: (912, 19)
[PREPROCESS] Categorical columns: ['gender', 'rgn']
[PREPROCESS] Task type: classification
[PREPROCESS] Missing values: 0 -> 0
[PREPROCESS] Final shape: (912, 19)
[PREPROCESS] Dataset identifier: pakistani-diabetes-dataset

PREPROCESSING COMPLETE
  Dataset identifier: pakistani-diabetes-dataset
  Processed shape: (912, 19)
  Target column: outcome
  Task type: classification
  Categorical columns: ['gender', 'rgn']
  Models to run: ['ctgan', 'tvae', 'copulagan', 'ctabgan', 'ctabganplus', 'pategan', 'medgan']
  RUN_MODE: debug
  N_TRIALS: 5
  Session: 2026-01-22
  Results path: results/pakistani-diabetes-dataset/2026-01-22/Section-2


### 2.3 Exploratory Data Analysis (EDA)

Comprehensive EDA with automated file export to results directory.

In [11]:
# Code Chunk ID: CHUNK_2_1_EDA - COMPREHENSIVE EDA
# ============================================================================
# Run comprehensive EDA with single function call
# Generates: column_analysis.csv, target_analysis.csv, target_balance_metrics.csv,
#            feature_distributions.png, correlation_heatmap.png, correlation_matrix.csv
# ============================================================================

from src.data.eda import run_comprehensive_eda

eda_results = run_comprehensive_eda(
    data=data,
    target_column=target_column,
    dataset_identifier=DATASET_IDENTIFIER,
    dataset_name=NOTEBOOK_CONFIG.get('dataset_name'),
    categorical_columns=categorical_columns,
    verbose=True
)

# Update categorical_columns from EDA results (in case auto-detected)
categorical_columns = eda_results['categorical_columns']

print(f"\nEDA Results Summary:")
print(f"  Files generated: {len(eda_results['files_generated'])}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Balance ratio: {eda_results.get('balance_ratio', 'N/A')}")

COMPREHENSIVE EXPLORATORY DATA ANALYSIS
Dataset: Pakistani Diabetes Dataset
Target column: outcome
Results path: results/pakistani-diabetes-dataset/2026-01-22/Section-2

[1/6] Dataset Overview...
   Dataset Name.................. Pakistani Diabetes Dataset
   Shape......................... 912 rows x 19 columns
   Memory Usage.................. 0.13 MB
   Total Missing Values.......... 0
   Missing Percentage............ 0.00%
   Duplicate Rows................ 2
   Numeric Columns............... 19
   Categorical Columns........... 0

[2/6] Column Analysis...
   Saved: column_analysis.csv (19 columns)

[3/6] Target Variable Analysis...
   Saved: target_analysis.csv
   Saved: target_balance_metrics.csv
   Balance Ratio: 0.877 (Balanced)

[4/6] Feature Distributions...
   Saved: 3 distribution file(s) (18 features)

[5/6] Correlation Analysis...
   Saved: correlation_heatmap.png
   Saved: correlation_matrix.csv
   Saved: target_correlations.csv (18 features)

[6/6] Configuration Validati

## 3 Demo All Models with Default Parameters

### 3.1 Batch Model Training

In [12]:
# Code Chunk ID: CHUNK_3_1_BATCH
# ============================================================================
# SECTION 3.1 - BATCH MODEL TRAINING
# Train all models configured in NOTEBOOK_CONFIG['models_to_run']
# ============================================================================

from src.models.batch_training import train_models_batch, extract_synthetic_data_to_globals

# Run batch training for all configured models
training_results = train_models_batch(
    data=data,
    models_to_run=models_to_run,
    target_column=target_column,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    verbose=True
)

# Extract synthetic_data_* variables for Section 3.2 compatibility
# This creates global variables like synthetic_data_ctgan, synthetic_data_tvae, etc.
created_vars = extract_synthetic_data_to_globals(training_results, globals())
print(f"\nCreated variables: {created_vars}")

# Also create model_* variables for backward compatibility
for model_name, result in training_results.items():
    if result['status'] == 'success':
        globals()[f'{model_name}_model'] = result['model']


BATCH MODEL TRAINING
Models to train: 7
Dataset shape: (912, 19)
Target column: outcome
Samples to generate: 912


[1/7] Training CTGAN...
--------------------------------------------------
  Training CTGAN...


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml fo

  Generating 912 synthetic samples...
  [OK] CTGAN completed in 17.33s
  Synthetic data shape: (912, 19)

[2/7] Training TVAE...
--------------------------------------------------
  Training TVAE...
  Generating 912 synthetic samples...
  [OK] TVAE completed in 11.75s
  Synthetic data shape: (912, 19)

[3/7] Training CopulaGAN...
--------------------------------------------------
  Training CopulaGAN...
  Generating 912 synthetic samples...
  [OK] CopulaGAN completed in 13.44s
  Synthetic data shape: (912, 19)

[4/7] Training CTABGAN...
--------------------------------------------------
  Training CTABGAN...


100%|██████████| 300/300 [00:33<00:00,  8.88it/s]


Finished training in 35.97795128822327  seconds.
  Generating 912 synthetic samples...
  [OK] CTABGAN completed in 36.07s
  Synthetic data shape: (912, 19)

[5/7] Training CTABGAN+...
--------------------------------------------------
  Training CTABGAN+...


100%|██████████| 400/400 [00:53<00:00,  7.44it/s]


Finished training in 55.945212602615356  seconds.
  Generating 912 synthetic samples...
  [OK] CTABGAN+ completed in 56.02s
  Synthetic data shape: (912, 19)

[6/7] Training PATE-GAN...
--------------------------------------------------
  Training PATE-GAN...
  Generating 912 synthetic samples...
  [OK] PATE-GAN completed in 9.87s
  Synthetic data shape: (912, 19)

[7/7] Training MEDGAN...
--------------------------------------------------
  Training MEDGAN...
  Generating 912 synthetic samples...
  [OK] MEDGAN completed in 12.06s
  Synthetic data shape: (912, 19)

BATCH TRAINING SUMMARY
Total models: 7
Successful: 7
Failed: 0

Successful models:
  - CTGAN: 17.33s
  - TVAE: 11.75s
  - CopulaGAN: 13.44s
  - CTABGAN: 36.07s
  - CTABGAN+: 56.02s
  - PATE-GAN: 9.87s
  - MEDGAN: 12.06s

Created variables: ['synthetic_data_ctgan', 'synthetic_data_tvae', 'synthetic_data_copulagan', 'synthetic_data_ctabgan', 'synthetic_data_ctabganplus', 'synthetic_data_pategan', 'synthetic_data_medgan']


### 3.2 Batch Evaluation

In [13]:
# Code Chunk ID: CHUNK_3_2_0_A
# ============================================================================
# SECTION 3.2 - BATCH EVALUATION FOR ALL TRAINED MODELS
# ============================================================================

print("SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION")
print("=" * 60)

section3_results = evaluate_all_available_models(
    section_number=3,
    scope=globals(),
    models_to_evaluate=None,
    real_data=None,
    target_col=None
)

if section3_results:
    print(f"\nSECTION 3 BATCH EVALUATION COMPLETED!")
    print(f"Evaluated {len(section3_results)} models successfully")
    
    # Show ranking by quality score
    best_models = []
    for model_name, results in section3_results.items():
        if 'error' not in results:
            quality_score = results.get('overall_quality_score', 0)
            best_models.append((model_name, quality_score))
    
    if best_models:
        best_models.sort(key=lambda x: x[1], reverse=True)
        print(f"\nRANKING BY QUALITY SCORE:")
        for i, (model, score) in enumerate(best_models, 1):
            print(f"   {i}. {model}: {score:.3f}")
else:
    print("\nNo models available for evaluation")

SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION
[OK] Batch evaluation functions loaded from src/evaluation/batch.py
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 3
[INFO] Dataset: pakistani-diabetes-dataset
[INFO] Target column: outcome
[INFO] Variable pattern: standard
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/pakistani-diabetes-dataset/2026-01-22/Section-3/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.786

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0.019
   [CHART] Explained Variance (PC1, PC2): 0.303, 0.085

[3] DI

## 4 Hyperparameter Tuning

### 4.1 Batch Hyperparameter Optimization

In [14]:
# Code Chunk ID: CHUNK_4_1_BATCH
# ============================================================================
# SECTION 4.1 - BATCH HYPERPARAMETER OPTIMIZATION
# ============================================================================

import sys
import importlib

# Ensure Optuna is installed
!{sys.executable} -m pip install -U optuna -q

import optuna
print(f"Optuna version: {optuna.__version__}")

# Reload modules to pick up optuna
import src.models.search_spaces as ss
ss = importlib.reload(ss)

import src.models.batch_optimization as bo
bo = importlib.reload(bo)

optimize_models_batch = bo.optimize_models_batch
extract_studies_to_globals = bo.extract_studies_to_globals

# Run batch optimization
optimization_results = optimize_models_batch(
    data=data,
    models_to_run=models_to_run,
    target_column=target_column,
    categorical_columns=categorical_columns,
    n_trials=N_TRIALS,
    run_mode=RUN_MODE,
    random_state=42,
    verbose=False
)

# Extract *_study variables for Section 4.2 compatibility
created_vars = extract_studies_to_globals(optimization_results, globals())
print(f"\nCreated study variables: {created_vars}")

[I 2026-01-22 17:51:23,812] A new study created in memory with name: ctgan_hpo


Optuna version: 4.7.0
[OK] Search spaces module loaded from src/models/search_spaces.py
[OK] Batch optimization module loaded from src/models/batch_optimization.py


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml fo

[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5303
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5707
[CHART] Combined Score: 0.5465 (Similarity: 0.5303, Accuracy: 0.5707)


Gen. (-1.67) | Discrim. (0.13): 100%|██████████| 250/250 [00:08<00:00, 28.23it/s] 


...

Finished training in 101.69805860519409  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5498
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9490
[CHART] Combined Score: 0.7095 (Similarity: 0.5498, Accuracy: 0.9490)


100%|██████████| 250/250 [00:28<00:00,  8.72it/s]


Finished training in 30.883084535598755  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6073
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9370
[CHART] Combined Score: 0.7391 (Similarity: 0.6073, Accuracy: 0.9370)


100%|██████████| 350/350 [01:38<00:00,  3.54it/s]


Finished training in 101.03856015205383  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5277
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9759
[CHART] Combined Score: 0.7069 (Similarity: 0.5277, Accuracy: 0.9759)
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.3108
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7336
[CHART] Combined Score: 0.4799 (Similarity: 0.3108, Accuracy: 0.7336)
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.2868
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7336
[CHART] Combined Score: 0.4655 (Similarity: 0.2868, Accuracy: 0.7336)
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.2935
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7336
[CHART] Comb

### 4.2 Save Best Parameters

In [15]:
# Code Chunk ID: CHUNK_4_1_SAVE
# ============================================================================
# SECTION 4.2 - SAVE BEST PARAMETERS TO CSV
# ============================================================================

from src.utils.parameters import save_best_parameters_to_csv

# Save all best parameters to CSV for Section 5
save_result = save_best_parameters_to_csv(
    scope=globals(),
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER
)

print(f"\nParameters saved: {save_result['success']}")
print(f"Files: {save_result.get('files_saved', [])}")

[SAVE] SAVING BEST PARAMETERS FROM SECTION 4
[FOLDER] Target directory: results/pakistani-diabetes-dataset/2026-01-22/Section-4

[CHART] Processing CTGAN parameters...
[OK] Found CTGAN: 10 parameters, score: 0.5465

[CHART] Processing CTAB-GAN parameters...
[OK] Found CTAB-GAN: 2 parameters, score: 0.7498

[CHART] Processing CTAB-GAN+ parameters...
[OK] Found CTAB-GAN+: 2 parameters, score: 0.7391

[CHART] Processing GANerAid parameters...
[WARNING]  GANerAid: Study variable 'ganeraid_study' not found

[CHART] Processing CopulaGAN parameters...
[OK] Found CopulaGAN: 6 parameters, score: 0.7030

[CHART] Processing TVAE parameters...
[OK] Found TVAE: 8 parameters, score: 0.7358

[OK] Parameters saved: results/pakistani-diabetes-dataset/2026-01-22/Section-4/best_parameters.csv
   - Total parameter entries: 34
[OK] Summary saved: results/pakistani-diabetes-dataset/2026-01-22/Section-4/best_parameters_summary.csv
   - Models processed: 5

[SAVE] Parameter saving completed!
[FOLDER] Files sa

### 4.3 Hyperparameter Optimization Batch Analysis

In [16]:
# Code Chunk ID: CHUNK_4_2_0_A
# ============================================================================
# SECTION 4.3 - HYPERPARAMETER OPTIMIZATION BATCH ANALYSIS
# ============================================================================

print("SECTION 4.3 - HYPERPARAMETER OPTIMIZATION BATCH ANALYSIS")
print("=" * 80)

# Build scope dict for evaluation
study_vars_found = sorted([k for k in globals().keys() if k.endswith("_study")])
print(f"Study vars found: {study_vars_found}")

scope_for_eval = dict(globals())

# Use batch evaluation function
try:
    section4_batch_results = evaluate_hyperparameter_optimization_results(
    section_number=4,
    scope=scope_for_eval,
    target_column=TARGET_COLUMN)
    
    print(f"\nSECTION 4 BATCH ANALYSIS COMPLETED!")
    print(f"Models analyzed: {section4_batch_results.get('models_analyzed', 0)}")
    
except Exception as e:
    print(f"Section 4 batch analysis error: {e}")
    import traceback
    traceback.print_exc()

SECTION 4.3 - HYPERPARAMETER OPTIMIZATION BATCH ANALYSIS
Study vars found: ['copulagan_study', 'ctabgan_study', 'ctabganplus_study', 'ctgan_study', 'medgan_study', 'pategan_study', 'tvae_study']
[LOCATION] Using DATASET_IDENTIFIER from scope: pakistani-diabetes-dataset
[TARGET] Final DATASET_IDENTIFIER for Section 4: pakistani-diabetes-dataset

SECTION 4 - HYPERPARAMETER OPTIMIZATION BATCH ANALYSIS
[FOLDER] Base results directory: results/pakistani-diabetes-dataset/2026-01-22/Section-4
[TARGET] Target column: outcome
[CHART] Dataset identifier: pakistani-diabetes-dataset


[SEARCH] 4.1.1: CTGAN Hyperparameter Optimization Analysis
------------------------------------------------------------
[OK] CTGAN optimization study found
[FOLDER] Model directory: results/pakistani-diabetes-dataset/2026-01-22/Section-4/CTGAN
[SEARCH] ANALYZING CTGAN HYPERPARAMETER OPTIMIZATION
[CHART] 1. TRIAL DATA EXTRACTION AND PROCESSING
----------------------------------------
[OK] Extracted 5 trials for analys

## 5 Final Model Comparison with Best Parameters

### 5.1 Train All Models with Best Parameters from Section 4

In [17]:
# Code Chunk ID: CHUNK_5_1_BATCH
# ============================================================================
# SECTION 5.1 - BATCH TRAINING WITH BEST PARAMETERS
# Replaces individual 5.1.1-5.1.6 model cells with single batch call
# ============================================================================

from src.models.batch_training import train_models_batch_with_best_params

# Train all models with best parameters from Section 4
# This creates synthetic_*_final variables automatically
section5_results = train_models_batch_with_best_params(
    data=data,
    target_column=target_column,
    models_to_run=models_to_run,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    section_number=4,  # Load params from Section 4
    dataset_identifier=DATASET_IDENTIFIER,
    scope=globals(),   # Creates synthetic_*_final variables
    verbose=True
)

# Show summary of created variables
final_vars = [k for k in globals().keys() if k.endswith('_final')]
print(f"\nFinal synthetic data variables: {sorted(final_vars)}")


SECTION 5.1: BATCH TRAINING WITH BEST PARAMETERS
Models to train: 7
Dataset shape: (912, 19)
Target column: outcome
Samples to generate: 912
Loading parameters from: Section 4

[1/3] Loading best parameters from Section 4...
[LOAD] LOADING BEST PARAMETERS FROM SECTION 4
[FOLDER] Looking for: results/pakistani-diabetes-dataset/2026-01-22/Section-4/best_parameters.csv
[OK] Found parameter CSV file
[OK] Loaded CTGAN: 10 parameters
[OK] Loaded CTAB-GAN: 2 parameters
[OK] Loaded CTAB-GAN+: 2 parameters
[OK] Loaded CopulaGAN: 6 parameters
[OK] Loaded TVAE: 8 parameters

[LOAD] Parameter loading completed!
[SEARCH] Source: CSV file
[CHART] Models loaded: 5
   - ctgan: 10 parameters
   - ctabgan: 2 parameters
   - ctabganplus: 2 parameters
   - copulagan: 6 parameters
   - tvae: 8 parameters
   Parameter source: CSV file
   Models with parameters: 5

[2/3] Training models with optimized parameters...

[1/7] Training CTGAN...
--------------------------------------------------
  Parameters load

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml fo

  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5171
[OK] TRTS Evaluation: 2 scenarios, Average: 0.6135
[CHART] Combined Score: 0.5557 (Similarity: 0.5171, Accuracy: 0.6135)
  [OK] CTGAN completed in 18.95s
  Synthetic data shape: (912, 19)
  Objective score: 0.5557

[2/7] Training TVAE...
--------------------------------------------------
  Parameters loaded: 8
    - epochs: 250
    - batch_size: 64
    - learning_rate: 0.0007
    - embedding_dim: 64
    - l2scale: 0.0000
    ... and 4 more
  Training TVAE...
  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5801
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9200
[CHART] Combined Score: 0.7161 (Similarity: 0.5801, Accuracy: 0.9200)
  [OK] TVAE completed in 10.80s
  Synthetic data shape: (912, 19)
  Objective 

100%|██████████| 300/300 [00:33<00:00,  8.89it/s]


Finished training in 35.96611475944519  seconds.
  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5633
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9731
[CHART] Combined Score: 0.7272 (Similarity: 0.5633, Accuracy: 0.9731)
  [OK] CTABGAN completed in 36.05s
  Synthetic data shape: (912, 19)
  Objective score: 0.7272

[5/7] Training CTABGAN+...
--------------------------------------------------
  Parameters loaded: 2
    - epochs: 250
    - batch_size: 512
    - categorical_columns: ['gender', 'rgn', 'outcome']
    - target_col: outcome
  Training CTABGAN+...


100%|██████████| 250/250 [00:28<00:00,  8.75it/s]


Finished training in 30.82048726081848  seconds.
  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6050
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9677
[CHART] Combined Score: 0.7501 (Similarity: 0.6050, Accuracy: 0.9677)
  [OK] CTABGAN+ completed in 30.90s
  Synthetic data shape: (912, 19)
  Objective score: 0.7501

[6/7] Training PATE-GAN...
--------------------------------------------------
  Parameters loaded: 0
    - discrete_columns: ['gender', 'rgn', 'outcome']
  Training PATE-GAN...
  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.3048
[ERROR] TRTS (Real->Synthetic) failed: Classification metrics can't handle a mix of continuous and binary targets
[ERROR] TRTS (Synthetic->Real) failed: Unknown label type: continuous. Maybe you are trying to fit a cla

### 5.2 Batch Evaluation of Optimized Models

In [18]:
# Code Chunk ID: CHUNK_5_2_0_A
# ============================================================================
# SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
# ============================================================================
!pip install --upgrade kaleido
print("SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION")
print("=" * 80)

from setup import evaluate_section5_optimized_models

try:
    section5_batch_results = evaluate_section5_optimized_models(
        section_number=5,
        scope=globals(),
        target_column=TARGET_COLUMN
    )
    
    print("\n" + "="*80)
    print("SECTION 5.2 OPTIMIZED MODELS BATCH EVALUATION COMPLETED!")
    print("="*80)
    print(f"Models processed: {section5_batch_results['models_processed']}")
    print(f"Results exported to: {section5_batch_results['results_dir']}")
    
    # Show summary
    if 'evaluation_summaries' in section5_batch_results:
        print("\nEVALUATION SUMMARIES:")
        print("-" * 40)
        for model_name, summary in section5_batch_results['evaluation_summaries'].items():
            print(f"{model_name}:")
            print(f"   Synthetic samples: {summary.get('synthetic_samples', 'N/A')}")
            print(f"   Overall score: {summary.get('overall_score', 'N/A')}")
            
except Exception as e:
    print(f"Section 5.2 batch evaluation failed: {e}")
    import traceback
    traceback.print_exc()

SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 5
[INFO] Dataset: pakistani-diabetes-dataset
[INFO] Target column: outcome
[INFO] Variable pattern: final
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/pakistani-diabetes-dataset/2026-01-22/Section-5/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.756

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0.030
   [CHART] Explained Variance (PC1, PC2): 0.303, 0.085

[3] DISTRIBUTION SIMILARITY
------------------------------
   [CHART] Aver

Traceback (most recent call last):
  File "/tmp/ipykernel_5957/53133864.py", line 21, in <module>
    print(f"Models processed: {section5_batch_results['models_processed']}")
KeyError: 'models_processed'


### 5.3 Final Summary

In [19]:
# Code Chunk ID: CHUNK_5_3_SUMMARY
# ============================================================================
# SECTION 5.3 - FINAL SUMMARY
# ============================================================================

print("=" * 80)
print("SYNTHETIC DATA GENERATION - FINAL SUMMARY")
print("=" * 80)
print(f"\nDataset: {DATASET_IDENTIFIER}")
print(f"Session: {SESSION_TIMESTAMP}")
print(f"\nResults directories:")
print(f"  Section 2 (EDA): {get_results_path(DATASET_IDENTIFIER, 2)}")
print(f"  Section 3 (Demo): {get_results_path(DATASET_IDENTIFIER, 3)}")
print(f"  Section 4 (HPO): {get_results_path(DATASET_IDENTIFIER, 4)}")
print(f"  Section 5 (Final): {get_results_path(DATASET_IDENTIFIER, 5)}")

# Show final model performance
if 'section5_results' in dir() and section5_results:
    print(f"\nFinal Model Performance (with optimized parameters):")
    print("-" * 60)
    
    scores = []
    for model_name, result in section5_results.items():
        if result['status'] == 'success':
            score = result.get('objective_score', 0)
            time_taken = result.get('training_time', 0)
            scores.append((model_name, score, time_taken))
    
    # Sort by score descending
    scores.sort(key=lambda x: x[1], reverse=True)
    
    for i, (model, score, time_taken) in enumerate(scores, 1):
        print(f"  {i}. {model.upper()}: score={score:.4f}, time={time_taken:.1f}s")
    
    if scores:
        best_model = scores[0][0]
        print(f"\nBest performing model: {best_model.upper()}")
        print(f"Best synthetic data variable: synthetic_{best_model}_final")

print("\n" + "=" * 80)
print("NOTEBOOK COMPLETE")
print("=" * 80)

SYNTHETIC DATA GENERATION - FINAL SUMMARY

Dataset: pakistani-diabetes-dataset
Session: 2026-01-22

Results directories:
  Section 2 (EDA): results/pakistani-diabetes-dataset/2026-01-22/Section-2
  Section 3 (Demo): results/pakistani-diabetes-dataset/2026-01-22/Section-3
  Section 4 (HPO): results/pakistani-diabetes-dataset/2026-01-22/Section-4
  Section 5 (Final): results/pakistani-diabetes-dataset/2026-01-22/Section-5

Final Model Performance (with optimized parameters):
------------------------------------------------------------
  1. CTABGANPLUS: score=0.7501, time=30.9s
  2. CTABGAN: score=0.7272, time=36.1s
  3. TVAE: score=0.7161, time=10.8s
  4. COPULAGAN: score=0.6772, time=68.1s
  5. CTGAN: score=0.5557, time=19.0s
  6. MEDGAN: score=0.4059, time=12.4s
  7. PATEGAN: score=0.3829, time=10.2s

Best performing model: CTABGANPLUS
Best synthetic data variable: synthetic_ctabganplus_final

NOTEBOOK COMPLETE
